# Bengaluru Climate Risk Scanner
## Report maps: flood (Phase 2) and heat (Phase 3)

Makes six print-ready landscape sheets, each with **two maps side by side**, from the saved outputs of Notebooks 02 and 03:

| Sheet | Left map | Right map |
|---|---|---|
| 1 | Height above nearest drainage | Slope |
| 2 | Distance to lakes | Distance to drains |
| 3 | Wetness index (TWI) | Built-up share |
| 4 | Flood risk index | Flood susceptibility classes |
| 5 | Summer surface temperature | Vegetation (NDVI) |
| 6 | Heat index | Extreme-heat classes |

Every map has a neat frame, a north arrow, a scale bar and a small legend. These sit in a panel at the right **inside** the frame, beside the city, so they never cover it. Sheets are saved as 300 dpi PNGs in `MyDrive/bengaluru-climate-risk/maps/`.

**Colours.** Flood maps run from cream through teal to deep navy; heat maps from pale yellow through orange to deep red; vegetation from pale yellow to deep green. Each ramp gets steadily darker, so it reads correctly for colour-blind readers and in black-and-white print. Lakes are a quiet blue-grey so they never look like a risk class.

It only reads files from Drive, so you can run it any time after Notebooks 02 and 03.

## 1. Connect Drive and load the configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)
OUT = CONFIG['paths']['processed']
GPKG = CONFIG['paths']['gpkg']
MAPS = f'{PROJECT_DIR}/maps'
os.makedirs(MAPS, exist_ok=True)

AUTHOR = 'Gowri'          # shown in the footer of every sheet; change to your full name
print('Maps will be saved to', MAPS)

## 2. Load the saved layers

In [ ]:
import numpy as np
import geopandas as gpd
import rasterio

def read(name, band=1):
    with rasterio.open(f'{OUT}/{name}') as src:
        a = src.read(band).astype('float32')
        if src.nodata is not None:
            a[a == src.nodata] = np.nan
        return a, src.transform, src.crs

fri, transform, crs = read('flood_risk.tif')
flood_class, _, _ = read('flood_class.tif')
F = {name: read('flood_factor_values.tif', i)[0]
     for i, name in enumerate(['hand', 'slope', 'dist_lake', 'dist_drain', 'twi', 'builtup'], start=1)}

heat, _, _ = read('heat_risk.tif')
heat_class, _, _ = read('heat_class.tif')
lst, _, _ = read('heat_values.tif', 1)
ndvi, _, _ = read('heat_values.tif', 2)

boundary = gpd.read_file(GPKG, layer='bbmp_boundary').to_crs(crs)
lakes = gpd.read_file(GPKG, layer='lakes').to_crs(crs)
lakes_city = gpd.clip(lakes, boundary.buffer(3000))

ny, nx = fri.shape
res = transform.a
EXTENT = [transform.c, transform.c + nx * res, transform.f - ny * res, transform.f]
city_mask = ~np.isnan(fri) | (flood_class == 0)
inside = lambda a: np.where(city_mask, a, np.nan)

FM, HM = CONFIG['flood_model'], CONFIG['heat_model']
print('Loaded. Grid', nx, 'x', ny, 'at', res, 'm')

## 3. Style and layout
- `sheet()` makes a landscape page with a double border, a title, two map frames and a footer.
- Each map frame is wider than the city: the extra space on the right (`SIDE`) is an empty panel inside the frame that holds the north arrow, scale bar and legend.
- `classed_panel()` draws maps with a few classes (legend of colour boxes); `continuous_panel()` draws smooth values (small colour bar).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.patches import Patch, Polygon, Rectangle

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
INK, MUTED, FRAME, PAPER, MAPBG = '#1f2328', '#6e7781', '#a0a8b0', 'white', '#f5f7f9'
LAKE, LAKE_EDGE = '#b7c7d6', '#8aa0b4'

FLOOD_5 = ['#fbf6d9', '#c4e6b8', '#5fbcc0', '#2b7bb3', '#16306b']   # cream -> teal -> navy
HEAT_5  = ['#fff3b8', '#fec65c', '#fd8a3c', '#e2412b', '#a50f26']   # pale yellow -> deep red
GREEN_5 = ['#fbf8d8', '#cfe8a3', '#8cc970', '#3f9a4b', '#0f5c2e']   # pale yellow -> deep green
FLOOD_3 = [FLOOD_5[1], FLOOD_5[2], FLOOD_5[4]]
HEAT_3  = [HEAT_5[1], HEAT_5[2], HEAT_5[4]]

SOURCES_FLOOD = 'Data: FABDEM (Copernicus GLO-30), ESA WorldCover 2021, OpenStreetMap contributors'
SOURCES_HEAT  = 'Data: Landsat 8/9 Collection 2 Level 2 (USGS), Mar–May 2022–2024, OpenStreetMap contributors'

xmin, ymin, xmax, ymax = boundary.total_bounds
W, H = xmax - xmin, ymax - ymin
SIDE = 0.62                       # side panel width, as a share of the city's width
VIEW = (xmin - 0.04 * W, xmax + SIDE * W, ymin - 0.05 * H, ymax + 0.05 * H)
BAND_X = (xmax + 0.04 * W - VIEW[0]) / (VIEW[1] - VIEW[0])   # where the side panel starts (axes fraction)

def north_arrow(ax, x=0.93, y=0.82, h=0.09, w=0.016):
    tip, base_l, base_r, notch = (x, y + h), (x - w, y), (x + w, y), (x, y + h * 0.28)
    ax.add_patch(Polygon([tip, base_l, notch], closed=True, transform=ax.transAxes, fc=INK, ec=INK, lw=0.7, zorder=10))
    ax.add_patch(Polygon([tip, notch, base_r], closed=True, transform=ax.transAxes, fc='white', ec=INK, lw=0.7, zorder=10))
    ax.text(x, y + h + 0.012, 'N', transform=ax.transAxes, ha='center', va='bottom', fontsize=9, fontweight='bold', color=INK)

def scale_bar(ax, y_frac=0.72, right_frac=0.965):
    x0, x1, y0, y1 = VIEW
    band_m = (x1 - x0) * (right_frac - BAND_X)
    length = next(L for L in (20, 10, 5, 2, 1) if L * 1000 <= band_m * 0.75)
    seg, hgt = length * 1000 / 4, (y1 - y0) * 0.012
    bx = x0 + (x1 - x0) * right_frac - length * 1000
    by = y0 + (y1 - y0) * y_frac
    for i in range(4):
        ax.add_patch(Rectangle((bx + i * seg, by), seg, hgt, fc=INK if i % 2 == 0 else 'white', ec=INK, lw=0.5, zorder=10))
    for frac, label in ((0, '0'), (0.5, f'{length / 2:g}'), (1, f'{length:g} km')):
        ax.text(bx + frac * length * 1000, by + hgt * 1.7, label, ha='center', va='bottom', fontsize=7, color=INK, zorder=10)

def sheet(title, subtitle, sources):
    fig = plt.figure(figsize=(13, 6.5), dpi=100, facecolor=PAPER)
    fig.add_artist(Rectangle((0.008, 0.012), 0.984, 0.976, transform=fig.transFigure, fill=False, ec=INK, lw=1.4))
    fig.add_artist(Rectangle((0.014, 0.022), 0.972, 0.956, transform=fig.transFigure, fill=False, ec=FRAME, lw=0.5))
    fig.text(0.035, 0.945, title, fontsize=17, fontweight='bold', color=INK, va='top')
    fig.text(0.035, 0.89, subtitle, fontsize=10, color=MUTED, va='top')
    axes = [fig.add_axes([0.035, 0.12, 0.455, 0.67]), fig.add_axes([0.51, 0.12, 0.455, 0.67])]
    for ax in axes:
        ax.set_xlim(VIEW[0], VIEW[1]); ax.set_ylim(VIEW[2], VIEW[3]); ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor(MAPBG)
        for s in ax.spines.values():
            s.set_edgecolor(FRAME); s.set_linewidth(0.8)
    fig.text(0.035, 0.065, sources, fontsize=7, color=MUTED)
    fig.text(0.035, 0.045, 'Projection: WGS 84 / UTM zone 43N (EPSG:32643)  ·  30 m grid', fontsize=7, color=MUTED)
    fig.text(0.965, 0.045, f'Bengaluru Climate Risk Scanner  ·  {AUTHOR}', fontsize=7, color=MUTED, ha='right')
    return fig, axes

def panel_title(ax, letter, name, note):
    ax.text(0, 1.065, f'{letter}   {name}', transform=ax.transAxes, fontsize=11.5, fontweight='bold', color=INK, va='bottom')
    ax.text(0, 1.015, note, transform=ax.transAxes, fontsize=8.5, color=MUTED, va='bottom')

def finish_panel(ax):
    if len(lakes_city):
        lakes_city.plot(ax=ax, color=LAKE, edgecolor=LAKE_EDGE, linewidth=0.2, zorder=4)
    boundary.boundary.plot(ax=ax, color=INK, linewidth=0.8, zorder=5)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xlim(VIEW[0], VIEW[1]); ax.set_ylim(VIEW[2], VIEW[3])
    north_arrow(ax); scale_bar(ax)

def classed_panel(ax, arr, colors, labels, legend_title):
    k = len(colors)
    ax.imshow(inside(arr), cmap=ListedColormap(colors), norm=BoundaryNorm(np.arange(0.5, k + 1), k),
              extent=EXTENT, interpolation='nearest', zorder=2)
    handles = [Patch(fc=c, ec=FRAME, lw=0.4, label=l) for c, l in zip(colors[::-1], labels[::-1])]
    handles.append(Patch(fc=LAKE, ec=LAKE_EDGE, lw=0.4, label='Lake / water body'))
    leg = ax.legend(handles=handles, title=legend_title, loc='lower right', bbox_to_anchor=(0.985, 0.03),
                    fontsize=7, title_fontsize=7.5, frameon=True, framealpha=1, facecolor='white',
                    edgecolor=FRAME, fancybox=False, handlelength=1.3, handleheight=0.9,
                    borderpad=0.6, labelspacing=0.32, alignment='left')
    leg.get_title().set_fontweight('bold'); leg.get_title().set_color(INK); leg.set_zorder(11)
    finish_panel(ax)

def continuous_panel(fig, ax, arr, colors, vmin, vmax, cbar_label, ticks):
    cmap = LinearSegmentedColormap.from_list('ramp', colors)
    im = ax.imshow(inside(arr), cmap=cmap, vmin=vmin, vmax=vmax, extent=EXTENT, interpolation='nearest', zorder=2)
    x0 = BAND_X + 0.03
    w = 0.955 - x0
    ax.add_patch(Rectangle((x0 - 0.015, 0.03), w + 0.03, 0.27, transform=ax.transAxes,
                           fc='white', ec=FRAME, lw=0.6, zorder=10))
    ax.text(x0, 0.255, cbar_label, transform=ax.transAxes, fontsize=7.5, fontweight='bold', color=INK, zorder=11)
    cax = ax.inset_axes([x0, 0.175, w, 0.045], zorder=11)
    cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=ticks)
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=6.5, length=2, width=0.4, colors=INK)
    ax.add_patch(Rectangle((x0, 0.06), 0.04, 0.035, transform=ax.transAxes, fc=LAKE, ec=LAKE_EDGE, lw=0.4, zorder=11))
    ax.text(x0 + 0.055, 0.0775, 'Lake / water body', transform=ax.transAxes, fontsize=7, color=INK, va='center', zorder=11)
    finish_panel(ax)

def save(fig, name):
    path = f'{MAPS}/{name}.png'
    fig.savefig(path, dpi=300, facecolor=PAPER)
    plt.show()
    print('Saved', path)

def ranges(breaks, unit, fmt='{:g}'):
    b = [fmt.format(x) for x in breaks]
    u = f' {unit}' if unit and unit not in ('%', '°') else unit
    return [f'< {b[0]}{u}'] + [f'{b[i]}–{b[i + 1]}{u}' for i in range(len(b) - 1)] + [f'≥ {b[-1]}{u}']

def score_classes(values, breaks, low_is_risky):
    idx = np.digitize(np.nan_to_num(values, nan=0), breaks)
    score = (5 - idx) if low_is_risky else (1 + idx)
    return np.where(np.isnan(values), np.nan, score).astype('float32')

def factor_panel(ax, letter, key, breaks, low_is_risky, unit, name, note, leg_title, fmt='{:g}'):
    labels = ranges(breaks, unit, fmt)
    by_score = labels[::-1] if low_is_risky else labels          # colour 1 (light) ... 5 (dark)
    by_score = [f'{l}   score {s}' for s, l in zip(range(1, 6), by_score)]
    classed_panel(ax, score_classes(F[key], breaks, low_is_risky), FLOOD_5, by_score, leg_title)
    panel_title(ax, letter, name, f'{note}  ·  weight {FM["weights_used"][key]:.2f}')

## 4. Phase 2: flood factors (sheets 1 to 3)
Each factor is drawn with the classes used for scoring, so the legend shows both the value range and the score it earns. Darkest = most flood-prone (score 5).

In [ ]:
fig, (a, b) = sheet('Flood factors: terrain', 'Low ground near drainage and flat land hold water', SOURCES_FLOOD)
factor_panel(a, 'a', 'hand', FM['hand_breaks_m'], True, 'm', 'Height above nearest drainage',
             'Metres above the stream, drain or lake each place drains to', 'HAND')
factor_panel(b, 'b', 'slope', FM['slope_breaks_deg'], True, '°', 'Slope', 'Flat ground drains slowly', 'Slope')
save(fig, 'sheet1_flood_terrain')

fig, (a, b) = sheet('Flood factors: water sources', 'Bengaluru floods start where lakes and stormwater drains overflow', SOURCES_FLOOD)
factor_panel(a, 'a', 'dist_lake', FM['lake_breaks_m'], True, 'm', 'Distance to lakes',
             'Straight-line distance to the nearest lake', 'Distance')
factor_panel(b, 'b', 'dist_drain', FM['drain_breaks_m'], True, 'm', 'Distance to drains',
             'Straight-line distance to the nearest drain or stream', 'Distance')
save(fig, 'sheet2_flood_water_sources')

fig, (a, b) = sheet('Flood factors: runoff', 'Where water gathers, and where concrete stops it soaking in', SOURCES_FLOOD)
factor_panel(a, 'a', 'twi', FM['twi_breaks'], False, '', 'Topographic wetness index',
             'Large drainage area on flat ground', 'TWI', fmt='{:.1f}')
factor_panel(b, 'b', 'builtup', [int(x * 100) for x in FM['builtup_breaks']], False, '%', 'Built-up share',
             'Share of each 30 m cell covered by buildings and roads', 'Built-up')
save(fig, 'sheet3_flood_runoff')

## 5. Phase 2: flood risk (sheet 4)

In [ ]:
fig, (a, b) = sheet('Flood susceptibility', 'Weighted overlay of six factors, classed by city percentiles', SOURCES_FLOOD)
continuous_panel(fig, a, fri, FLOOD_5, 1, 5, 'Flood risk index', ticks=[1, 2, 3, 4, 5])
panel_title(a, 'a', 'Flood risk index', '1 = safe, 5 = very flood-prone')

areas = {c: np.nansum(flood_class == c) * res * res / 1e6 for c in (1, 2, 3)}
tot = sum(areas.values())
lab = [f'{l}  {areas[c]:.0f} km² ({100 * areas[c] / tot:.0f}%)' for c, l in zip((1, 2, 3), ('Low', 'Medium', 'High'))]
classed_panel(b, np.where(flood_class == 0, np.nan, flood_class), FLOOD_3, lab, 'Flood risk')
panel_title(b, 'b', 'Flood classes', f'Breaks at FRI {FM["class_breaks_fri"][0]} and {FM["class_breaks_fri"][1]}')
save(fig, 'sheet4_flood_risk')

## 6. Phase 3: summer heat (sheets 5 and 6)

In [ ]:
lo, hi = np.floor(np.nanpercentile(lst, 2)), np.ceil(np.nanpercentile(lst, 98))
fig, (a, b) = sheet('Summer surface heat', 'Median of clear Landsat 8/9 scenes, March–May 2022–2024', SOURCES_HEAT)
continuous_panel(fig, a, lst, HEAT_5, lo, hi, 'Surface temperature (°C)', ticks=np.linspace(lo, hi, 4).round(0))
panel_title(a, 'a', 'Land surface temperature', 'Typical hot-season surface temperature')
continuous_panel(fig, b, ndvi, GREEN_5, 0, 0.6, 'NDVI', ticks=[0, 0.2, 0.4, 0.6])
panel_title(b, 'b', 'Vegetation (NDVI)', 'Greener surfaces stay cooler')
save(fig, 'sheet5_heat_inputs')

areas = {c: np.nansum(heat_class == c) * res * res / 1e6 for c in (1, 2, 3)}
mean_t = {c: np.nanmean(lst[heat_class == c]) for c in (1, 2, 3)}
lab = [f'{l}  {areas[c]:.0f} km² · {mean_t[c]:.1f} °C' for c, l in zip((1, 2, 3), ('Low', 'Medium', 'High'))]
uhi = HM.get('surface_uhi_c')
fig, (a, b) = sheet('Extreme-heat risk',
                    '0.7 × normalised temperature + 0.3 × (1 − normalised NDVI)' +
                    (f'  ·  surface urban heat island {uhi:+.1f} °C' if uhi is not None else ''), SOURCES_HEAT)
continuous_panel(fig, a, heat, HEAT_5, 0, 1, 'Heat index', ticks=[0, 0.5, 1])
panel_title(a, 'a', 'Heat index', '0 = coolest, 1 = hottest and least green')
classed_panel(b, np.where(heat_class == 0, np.nan, heat_class), HEAT_3, lab, 'Heat risk · area · LST')
panel_title(b, 'b', 'Heat classes', 'Classed by city percentiles of the heat index')
save(fig, 'sheet6_heat_risk')

## 7. Files saved

In [ ]:
for f in sorted(os.listdir(MAPS)):
    print(f'  {f:<32} {os.path.getsize(f"{MAPS}/{f}") / 1e6:5.1f} MB')